<div style="padding: 28px 32px; border-radius: 18px; background: linear-gradient(135deg, #111827 0%, #0f766e 48%, #22c55e 100%); color: white; box-shadow: 0 20px 50px rgba(17, 24, 39, 0.18);">
  <p style="margin: 0; font-size: 14px; letter-spacing: 0.18em; text-transform: uppercase; opacity: 0.85;">Customer Intelligence • Revenue Forecasting System</p>
  <h1 style="margin: 12px 0 10px; font-size: 34px;">RFM Customer Segmentation</h1>
  <p style="margin: 0; font-size: 17px; max-width: 840px; line-height: 1.7;">
    This notebook translates transaction history into customer-level recency, frequency, and monetary signals to support targeting, retention, and value-based customer strategy.
  </p>
</div>


In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from matplotlib.ticker import FuncFormatter

pd.options.display.float_format = "{:,.2f}".format
sns.set_theme(style="whitegrid", context="talk")

project_root = Path.cwd()
if not (project_root / "data").exists() and (project_root.parent / "data").exists():
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from scripts.data_utils import load_cleaned_data
from scripts.rfm_utils import build_rfm_table

def currency(x, pos=None):
    return f"${x:,.0f}"

df = load_cleaned_data(project_root)
rfm = build_rfm_table(df)
rfm.head()


## Why RFM Matters


RFM helps companies prioritize retention and growth actions using actual purchase behavior:

- **Recency** shows how fresh the relationship is.
- **Frequency** reflects habitual buying behavior.
- **Monetary** reveals commercial value.

These signals are practical for marketing, loyalty strategy, account prioritization, and dashboard storytelling.


## Segment Distribution


In [ ]:
segment_counts = rfm["rfm_segment"].value_counts().reset_index()
segment_counts.columns = ["rfm_segment", "customer_count"]
segment_counts


In [ ]:
plt.figure(figsize=(12, 6))
ax = sns.barplot(data=segment_counts, x="rfm_segment", y="customer_count", hue="rfm_segment", legend=False, palette="crest")
ax.set_title("Customer Count by RFM Segment")
ax.set_xlabel("")
ax.set_ylabel("Customers")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


## Segment Value Analysis


In [ ]:
segment_analysis = (
    rfm.groupby("rfm_segment")
    .agg(
        customers=("customer_id", "nunique"),
        avg_spending=("monetary", "mean"),
        total_revenue=("monetary", "sum"),
        avg_recency=("recency", "mean"),
    )
    .reset_index()
    .sort_values("total_revenue", ascending=False)
)
segment_analysis["revenue_contribution_pct"] = (
    segment_analysis["total_revenue"] / segment_analysis["total_revenue"].sum()
)
segment_analysis


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.barplot(data=segment_analysis, x="rfm_segment", y="avg_spending", hue="rfm_segment", legend=False, palette="Blues", ax=axes[0])
axes[0].set_title("Average Spending by RFM Segment")
axes[0].tick_params(axis="x", rotation=20)
axes[0].yaxis.set_major_formatter(FuncFormatter(currency))

sns.barplot(data=segment_analysis, x="rfm_segment", y="total_revenue", hue="rfm_segment", legend=False, palette="Greens", ax=axes[1])
axes[1].set_title("Revenue Contribution by RFM Segment")
axes[1].tick_params(axis="x", rotation=20)
axes[1].yaxis.set_major_formatter(FuncFormatter(currency))

plt.tight_layout()
plt.show()


> **Insight:** RFM segmentation turns customer history into action. High-value segments deserve retention investment, while aging segments highlight where reactivation campaigns can create measurable upside.


## Save RFM Output


In [ ]:
output_path = project_root / "outputs" / "csv" / "rfm_table.csv"
processed_path = project_root / "data" / "processed" / "customer_level_features.csv"

output_path.parent.mkdir(parents=True, exist_ok=True)
processed_path.parent.mkdir(parents=True, exist_ok=True)

rfm.to_csv(output_path, index=False)
rfm.to_csv(processed_path, index=False)

print(f"Saved RFM output to: {output_path.resolve()}")


## Dashboard Usage

This table will feed both Power BI and the Streamlit dashboard for segment counts, segment revenue, customer tables, and customer prioritization views.
